In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:1b")

# 이게 프롬프트 -> LLM을 호출할 때 사용하는 명령어라고 할 수 있음
# shell command와 유사
llm.invoke(1)
# 이것도 명령어여서 ValueError: Invalid input type <class 'int'>. Must be a PromptValue, str, or list of BaseMessages.
# PromptValue -> langchain에 PromptTemplate이 있는데 이걸 쓰면 만들 수 있음

ValueError: Invalid input type <class 'int'>. Must be a PromptValue, str, or list of BaseMessages.

In [ ]:
from langchain_core.prompts  import PromptTemplate

prompt_template = PromptTemplate(
    template="What is the capital of {country}?",
    input_variables=["country"]
)

# (variable) prompt: PromptValue
prompt = prompt_template.invoke({"country": "France"})
# invoke를 해서 가져올 수 있다
llm.invoke(prompt)  # 세미콜론(;) 있으면 Jupyter에서 반환값이 안 보임
# 결국 이거는 llm.invoke(prompt_template.invoke({"country": "France"}))와 같음

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-02-22T08:02:14.600846Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1111245667, 'load_duration': 112653125, 'prompt_eval_count': 32, 'prompt_eval_duration': 812510167, 'eval_count': 8, 'eval_duration': 175869082, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019c845e-d55d-72a3-a566-22cb0f9f51a7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [ ]:
# BaseMessages list
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
# 이때 HumanMessage는 BaseMessage를 상속받음 -> 사용자가 입력한 메시지를 나타내는 클래스
# BaseMessage를 상속받는 대표적 4개의 클래스
# 1. system: 우리가 만드는 LLM app의 목적(페르소나 - 이 어플리케이션이 할 일을 정해주는 것)
# 2. human: 사용자가 입력한 메시지
# 3. ai: LLM이 생성한 메시지
# 4. tool: 도구()
# 이 리스트들도 llm invoke에 넣을 수 있음

# ValueError: Invalid input type <class 'langchain_core.messages.human.HumanMessage'>. Must be a PromptValue, str, or list of BaseMessages.
# llm.invoke(HumanMessage(content="What is the capital of France?"))

# 반드시 리스트여야 함 아니면 에러 발생
llm.invoke([HumanMessage(content="What is the capital of France?")])

country = 'China'  # HumanMessage f-string에서 사용할 변수
messages = [
    SystemMessage(content="You are a helpful assistant that can answer questions."),
    HumanMessage(content="What is the capital of France?"),
    # 이때 AIMessage는 few shot와 같음 답변 형식을 정해주는 것
    # 마치 대화 이력이 있던 것처럼 llm을 속여서 답변을 우리가 원하는대로 하도록 유도하는 것
    # check few shot learner -> 복잡한 문제는 예제를 주면 정확도가 높아짐 그럴 때 이렇게 리스트로 주면 좋음
    AIMessage(content="The capital of France is Paris."),
    # ToolMessage는 tool_call_id 필수 (어떤 도구 호출에 대한 결과인지 구분용)
    # ToolMessage(content="I'm a tool message.", tool_call_id="call_example_123"),
    HumanMessage(content=f"What is the capital of {country}?"),
]

llm.invoke(messages)

AIMessage(content="You're asking me to specify which country or region you're interested in! There are several capitals around the world, so I'll give you a few examples:\n\n* The capital of France is Paris.\n* The capital of China is Beijing.\n* The capital of India is New Delhi.\n* The capital of Brazil is Brasília.\n\nLet me know if you have a specific country or region in mind, and I'll do my best to provide the correct answer!", additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-02-22T08:23:32.00399Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3155970583, 'load_duration': 114022875, 'prompt_eval_count': 65, 'prompt_eval_duration': 102218667, 'eval_count': 93, 'eval_duration': 2752131292, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019c8472-4b4e-7022-9f02-644a7eb81f5e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 93, 'total_tokens': 158})

In [ ]:
# 강사는 근데 이런 방법은 langChain스럽지 않다고 생각함
from langchain_core.prompts import ChatPromptTemplate
# 채팅 메시지처럼 쓰는 방법

# 대신 여기서 사용할 떄는 튜플로 사용해야 됨
chat_prompt_template = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant that can answer questions.'), 
    ('human', 'What is the capital of {country}?')
])

chat_prompt = chat_prompt_template.invoke({"country": "Italy"})

print(chat_prompt)


messages=[SystemMessage(content='You are a helpful assistant that can answer questions.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={})]


In [39]:
chat_prompt.messages

[SystemMessage(content='You are a helpful assistant that can answer questions.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={})]

In [40]:
llm.invoke(chat_prompt)

AIMessage(content='The capital of Italy is Rome.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-02-22T08:25:32.814194Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1548369333, 'load_duration': 103429250, 'prompt_eval_count': 42, 'prompt_eval_duration': 1205571583, 'eval_count': 8, 'eval_duration': 209732914, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019c8474-2974-7c72-bb9d-5948ee1f229b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})

그래서 대화 이력을 넣어준다고 하면
messages보다 아래 방식이 더 langchain 스럽다고 생각한다
-> LCEL
: langchain 구성 요소를 연결
이때 PromptTemplate은 연동이 되는데 list of BaseMessages는 여기 낄 수가 없음

그래서 이 방식이 확장할 떄 더 유리하기 떄문에 프롬프트 템플릿 사용이 더 유리함
